# Bloomberg Terminal Connectivity Test

This document serves as the preliminary technical validation for the data extraction layer. The objective is to verify the low-level API linkage with the Bloomberg Terminal (`bbcomm`) before proceeding with the formal data ingestion pipelines.

## Validation Scope

1. Verification of the `xbbg` and `blpapi` underlying bindings.
2. Confirmation of purely vectorized extraction, bypassing iterative procedures and yielding $\mathcal{O}(1)$ asymptotic time complexity over the time-series arrays.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure the project root is in the system path to permit absolute imports from the 'src' package robustly.
current_dir = Path(os.path.abspath(''))
project_root = current_dir
while not (project_root / 'src').exists() and project_root.parent != project_root:
    if (project_root / 'Algorithmic_Trading_Backtester' / 'src').exists():
        project_root = project_root / 'Algorithmic_Trading_Backtester'
        break
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.infrastructure.BloombergDataHandler import BloombergDataHandler


## Execution Routine

The extraction focuses on acquiring the most recent data matrices. In an active production environment, the `api_connection` should bind directly to the Bloomberg session. For academic and isolated testing, mock environments are supported.

In [ ]:
def test_bloomberg_connectivity() -> None:
    """
    Executes an isolated connection test to the Bloomberg Terminal.
    """
    print("Initiating connection test with Bloomberg Terminal...")
    
    # Note: In a live environment, a real connection object must be injected.
    class MockApiConnection:
        pass

    try:
        handler = BloombergDataHandler(
            api_connection=MockApiConnection(), 
            start_date='2023-01-01', 
            end_date='2023-01-15'
        )
        
        # The load_data method operates under O(1) loop logic constraint.
        df = handler.load_data(symbol='AAPL US Equity')
        
        if df is None or df.empty:
            print("[WARNING] Connection succeeded, but no data was retrieved. Check ticker validity or API limits.")
        else:
            print("[SUCCESS] Data extracted and vectorized into an O(1) DataFrame layout successfully:")
            print("-" * 50)
            print(df.head())
            print("-" * 50)
            print(f"In-memory matrix dimensions: {df.shape}")
            
    except ImportError as e:
        print("[CONNECTION FAILURE] Cannot communicate with bbcomm or the xbbg library is missing.")
        print(f"Technical Error: {e}")
        print("\nEnsure that:")
        print("1. The Bloomberg Terminal session is active.")
        print("2. The bbcomm.exe process is running.")
        print("3. blpapi and xbbg packages are installed.")

# Execute the routine
test_bloomberg_connectivity()
